chroma는 인메모리인 반면
pinecone는 클라우드를 활용

0. 문서는 워드로 추천(한글은 못읽고 PDF는 PDF로더로 가능은 하나 줄바꿈 시 인식오류가 발생함)
 - 문서 포맷이 .docx 가 아닌 경우 반드시 .docx로 변경
1. 워드 문서의 내용을 읽는다.
2. 문서를 쪼갠다(chunk)
    - 토근수 초과로 답볍을 생성하지 못할수 있고
    - 문서가 길명(인풋이 길면) 답변 생성이 오래걸림
3. 임베딩 => 벡터 데이터베이스에 저장
4. 질문이 있을때, 벡터 데이터베이스에 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달

In [2]:
%pip install python-dotenv langchain langchain-upstage langchain-community langchain-text-splitters docx2txt langchain-chroma

  Using cached python_dotenv-1.2.2-py3-none-any.whl.metadata (27 kB)
  Using cached langchain_upstage-0.7.7-py3-none-any.whl.metadata (3.3 kB)
  Using cached langchain_community-0.4.2-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
  Using cached docx2txt-0.9-py3-none-any.whl.metadata (529 bytes)
  Using cached langchain_chroma-1.1.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached langchain_core-1.4.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached pydantic-2.13.4-py3-none-any.whl.metadata (109 kB)
  Using cached langchain_openai-1.2.2-py3-none-any.whl.metadata (3.1 kB)
  Using cached pypdf-6.12.2-py3-none-any.whl.metadata (7.2 kB)
  Using cached tokenizers-0.20.3-cp312-none-win_amd64.whl.metadata (6.9 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached langchain_classic-1.0.7-py3-none-any.whl.metadata (5.1 kB)
  Using cached pydantic_settings-2.14.1-py3-none-any.whl.metadata (3.4 k

  You can safely remove it manually.
  You can safely remove it manually.

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
%pip install langchain langchain-core langchain-community langchain-text-splitters langchain-openai langchain-pinecone docx2txt

  Using cached langchain_pinecone-0.2.13-py3-none-any.whl.metadata (8.6 kB)
  Using cached pinecone-7.3.0-py3-none-any.whl.metadata (9.5 kB)
  Using cached simsimd-6.5.16-cp312-cp312-win_amd64.whl.metadata (71 kB)
  Using cached pinecone_plugin_assistant-1.8.0-py3-none-any.whl.metadata (30 kB)
  Using cached pinecone_plugin_interface-0.0.7-py3-none-any.whl.metadata (1.2 kB)
  Using cached aiohttp_retry-2.9.1-py3-none-any.whl.metadata (8.8 kB)
  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)
Using cached langchain_pinecone-0.2.13-py3-none-any.whl (26 kB)
Using cached pinecone-7.3.0-py3-none-any.whl (587 kB)
Using cached simsimd-6.5.16-cp312-cp312-win_amd64.whl (87 kB)
Using cached aiohttp_retry-2.9.1-py3-none-any.whl (10.0 kB)
Using cached pinecone_plugin_assistant-1.8.0-py3-none-any.whl (259 kB)
Using cached packaging-24.2-py3-none-any.whl (65 kB)
Using cached pinecone_plugin_interface-0.0.7-py3-none-any.whl (6.2 kB)
  Attempting uninstall: packaging
    Found existing 


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from langchain_upstage import UpstageEmbeddings
from dotenv import load_dotenv
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain_upstage import ChatUpstage
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate


load_dotenv()

embedding = UpstageEmbeddings(model='solar-embedding-1-large')

index_name = "tax-index-4096"

llm = ChatUpstage()
database = PineconeVectorStore.from_existing_index(index_name, embedding)

query = '연봉 5천만원인 거주자의 종합소득세는 얼마인가요?'
retrieved_docs = database.similarity_search(query, k=3) 

prompt = PromptTemplate.from_template("""
You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Question: {question}

Context: {context}

Answer:
""")

retriever=database.as_retriever()

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=database.as_retriever(),
    chain_type_kwargs={"prompt": prompt},
)

ai_message =  qa_chain.invoke({"query":query})

dictionary = ["사람을 나타내는 표현 -> 거주자"]

prompt = ChatPromptTemplate.from_template(f"""
    사용자의 질문을 보고, 우리의 사전을 참고해서 사용자의 질문을 변경해주세요.
    만약 변경할 필요가 없다고 판단된다면, 사용자의 질문을 변경하지 않아도 됩니다.
    그런 경우에는 질문만 리턴해주세요
    사전: {dictionary}
    
    질문: {{question}}
""")

dictionary_chain = prompt | llm | StrOutputParser()
tax_chain = {"query": dictionary_chain} | qa_chain

ai_response =  tax_chain.invoke({"question":query})

In [ ]:
%pip install -U langchain langchainhub --quiet

In [ ]:
prompt

In [ ]:
# %pip install -U langchain-classic

In [ ]:
ai_message

In [ ]:
query

In [ ]:
ai_response